# Genesis World — Franka grasps a soft body (Day36, Step 4)

Part of the `surgical-video-ai-learning` Day36 exploration: can Genesis pseudo-model a surgical field?
Steps 1–3 and 5–6 (cloth draping, adhesion, tethered soft body, multi-body pile, bounded cavity) ran fine on CPU
(see `playground/genesis-world/` in the repo). This one — a robot arm (Franka) grasping and lifting a soft
(deformable, MPM) cube — is the closest analogy to **an instrument grasping tissue**, but the official example
assumes a GPU and uses inverse kinematics (IK) control, which is slow on a CPU laptop. Colab's free T4 GPU is a
better fit, so this step is parked here instead of the local `.venv`.

**Before running:** `Runtime > Change runtime type > T4 GPU` (or any available GPU).

Based on Genesis's official `examples/coupling/grasp_soft_cube.py`, adapted to:
- run headless (no interactive viewer — Colab has no display), using an offscreen camera instead
- save the result as a GIF you can view inline or download

**Physics concept:** contact-force grasping. A rigid gripper doesn't just teleport the soft cube — it has to
apply enough squeeze force (via friction at the finger contacts) to counteract gravity while the cube itself
deforms under that grip. Too little grip force and it slips; the cube's own softness means "holding on" looks
different from gripping a rigid object.

## 1. Install

Genesis needs Python ≥3.10 (Colab's default is fine) and a CUDA-enabled `torch` (Colab's preinstalled torch
already matches its GPU, so we leave torch alone and only add `genesis-world` + `imageio`).

In [ ]:
!pip install -q genesis-world imageio

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

If `CUDA available: False`, go to `Runtime > Change runtime type` and select a GPU, then re-run this cell.

In [ ]:
import numpy as np
import genesis as gs

gs.init(backend=gs.gpu)

## 2. Build the scene

- `plane`: the ground
- `cube`: a small soft (`MPM.Elastic`) cube — the "tissue" being grasped
- `franka`: a Franka Emika Panda arm (bundled with Genesis's assets, same as the CPU-side scripts' `xml/franka_emika_panda/panda.xml`)

In [ ]:
scene = gs.Scene(
    show_viewer=False,
    sim_options=gs.options.SimOptions(dt=5e-3, substeps=15),
    mpm_options=gs.options.MPMOptions(
        lower_bound=(0.55, -0.1, -0.05),
        upper_bound=(0.75, 0.1, 0.3),
        grid_density=128,
    ),
    vis_options=gs.options.VisOptions(
        ambient_light=(0.6, 0.6, 0.6),
        background_color=(0.9, 0.9, 0.9),
        visualize_mpm_boundary=True,
    ),
)

plane = scene.add_entity(gs.morphs.Plane())

cube = scene.add_entity(
    material=gs.materials.MPM.Elastic(),
    morph=gs.morphs.Box(
        size=(0.04, 0.04, 0.04),
        pos=(0.65, 0.0, 0.025),
        euler=(0, 0, 0),
    ),
    surface=gs.surfaces.Default(color=(0.85, 0.35, 0.3, 1.0), vis_mode="particle"),
)

franka = scene.add_entity(
    gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"),
    material=gs.materials.Rigid(coup_friction=1.0),
)

camera = scene.add_camera(
    res=(480, 360),
    pos=(1.1, -0.7, 0.6),
    lookat=(0.65, 0.0, 0.15),
    fov=35,
)

scene.build()

## 3. Control setup

Position/velocity gains and force limits for the arm's 9 DOFs (7 arm joints + 2 gripper fingers), taken directly
from the official example.

In [ ]:
motors_dof = np.arange(7)
fingers_dof = np.arange(7, 9)

franka.set_dofs_kp(np.array([4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100]))
franka.set_dofs_kv(np.array([450, 450, 350, 350, 200, 200, 200, 10, 10]))
franka.set_dofs_force_range(
    np.array([-87, -87, -87, -87, -12, -12, -12, -100, -100]),
    np.array([87, 87, 87, 87, 12, 12, 12, 100, 100]),
)

end_effector = franka.get_link("hand")

# pre-grasp pose, fingers open
qpos = franka.inverse_kinematics(
    link=end_effector,
    pos=np.array([0.64, 0.0, 0.135]),
    quat=np.array([0, 1, 0, 0]),
)
qpos[-2:] = 0.03
franka.set_dofs_position(qpos[:-2], motors_dof)
franka.set_dofs_position(qpos[-2:], fingers_dof)

## 4. Grasp and lift, recording frames

Three phases: close the fingers with 1N of force (grasp), raise the end effector while holding that grip force
(lift), then hold in place (settle). A frame is rendered every few steps so the GIF stays a reasonable size.

In [ ]:
frames = []
render_every = 4

# grasp with 1N force
franka.control_dofs_position(qpos[:-2], motors_dof)
franka.control_dofs_force(np.array([-1, -1]), fingers_dof)

for i in range(100):
    scene.step()
    if i % render_every == 0:
        rgb, _, _, _ = camera.render()
        frames.append(rgb)

# lift
for i in range(300):
    qpos = franka.inverse_kinematics(
        link=end_effector,
        pos=np.array([0.64, 0.0, 0.135 + 0.0005 * i]),
        quat=np.array([0, 1, 0, 0]),
    )
    franka.control_dofs_position(qpos[:-2], motors_dof)
    scene.step()
    if i % render_every == 0:
        rgb, _, _, _ = camera.render()
        frames.append(rgb)

# hold
for i in range(100):
    scene.step()
    if i % render_every == 0:
        rgb, _, _, _ = camera.render()
        frames.append(rgb)

print(f"Recorded {len(frames)} frames")

## 5. Save and view the GIF

In [ ]:
import imageio

imageio.mimsave("grasp_soft_body.gif", frames, duration=0.05)

from IPython.display import Image as IPyImage, display
display(IPyImage(filename="grasp_soft_body.gif"))

In [ ]:
# Optional: download the GIF to bring back into the surgical-video-ai-learning repo
from google.colab import files
files.download("grasp_soft_body.gif")

## What to look for

- Does the cube stay gripped through the lift, or slip out of the fingers? (If it slips, `-1N` finger force may not
  be enough for this cube's default material stiffness/friction — try a stronger force, e.g. `-2` to `-5`.)
- Does the cube visibly **deform** where the fingers press into it, unlike a rigid object which would just be
  displaced with no shape change? That deformation is the MPM solver, not the gripper animation.
- Surgical framing: this is the closest of the six Day36 experiments to actual instrument-tissue interaction —
  grip force has to be enough to hold on, but too much would (in principle, if the material model supports it)
  crush or tear the tissue.